# Wikidata Airport Extraction & Fusion

**Purpose**: pull the raw attribute values profiled in `wikidata_profiling.ipynb`, resolve any per-airport conflicts down to a single trustworthy value per attribute (in pandas, not SPARQL), and merge everything into one final Wikidata airport dataset.

**Pipeline**:
1. Fetch raw, unaggregated `(airport, name, value)` rows per attribute (one query per attribute — combining them would create a cartesian product, see `wikidata_profiling.ipynb`) and save them to `data/raw/wikidata/`.
2. Resolve conflicts per attribute in pandas.
3. Merge into one wide table, keyed by `airport`, and save to `data/interim/wikidata_airports.csv`.

This notebook does not do exploratory analysis on the *result* — that's `wikidata_eda.ipynb`, which reads the file this notebook produces.

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from wdi_airports.ufreiburg_sparql import run_sparql_query

WIKIDATA_ENDPOINT = "https://qlever.dev/api/wikidata"
RAW_DIR = project_root / "data" / "raw" / "wikidata"
INTERIM_DIR = project_root / "data" / "interim"

## 1. Raw extraction

Same attribute list as `wikidata_profiling.ipynb`, but now pulling actual values instead of just counting them. Each query returns raw `(airport, name, value)` rows with **no** `GROUP BY` / aggregation — an airport with 2 conflicting values just produces 2 rows, and we resolve that in pandas afterwards.

In [2]:
LITERAL_ATTRIBUTES = {
    "iata": "wdt:P238",
    "icao": "wdt:P239",
    "faa": "wdt:P240",
}

# These attributes point to another Wikidata entity (a place or organization), not a
# plain literal -- resolved to the entity's English label so the dataset holds readable
# text instead of a Q-ID URI. If an entity has no English label, fall back to its raw
# URI (COALESCE) rather than silently losing the value.
ENTITY_ATTRIBUTES = {
    "located_in": "wdt:P131",
    "country": "wdt:P17",
    "operator": "wdt:P137",
    "owner": "wdt:P127",
}

# Airport class filter includes subclasses (international/commercial/domestic airport, ...) --
# see wikidata_profiling.ipynb: the strict wdt:P31 wd:Q1248784 (no subclasses) silently
# excluded ~636 real airports, including Berlin Brandenburg and Berlin Schönefeld.
build_literal_query = lambda propName: f"""
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?airport ?name ?value
WHERE {{
  ?airport wdt:P31/wdt:P279* wd:Q1248784 ;
           rdfs:label ?name .
  FILTER(LANG(?name) = "en")
  OPTIONAL {{ ?airport {propName} ?value }}
}}
"""

build_entity_query = lambda propName: f"""
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?airport ?name ?value
WHERE {{
  ?airport wdt:P31/wdt:P279* wd:Q1248784 ;
           rdfs:label ?name .
  FILTER(LANG(?name) = "en")
  OPTIONAL {{
    ?airport {propName} ?entity .
    OPTIONAL {{ ?entity rdfs:label ?label . FILTER(LANG(?label) = "en") }}
    BIND(COALESCE(?label, STR(?entity)) AS ?value)
  }}
}}
"""

# Coordinates: lat/lon both come from the same wdt:P625 statement, so fetching them
# together in one OPTIONAL doesn't create a cartesian product (they're paired, not independent).
COORD_QUERY = """
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX geof: <http://www.opengis.net/def/function/geosparql/>

SELECT ?airport ?name ?lat ?lon
WHERE {
  ?airport wdt:P31/wdt:P279* wd:Q1248784 ;
           rdfs:label ?name .
  FILTER(LANG(?name) = "en")
  OPTIONAL {
    ?airport wdt:P625 ?coord .
    BIND(geof:latitude(?coord) AS ?lat)
    BIND(geof:longitude(?coord) AS ?lon)
  }
}
"""

# Timezone: two-hop, see wikidata_profiling.ipynb for why (airports have no direct wdt:P421).
# Also resolved to a label, same reasoning as ENTITY_ATTRIBUTES above.
TIMEZONE_QUERY = """
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?airport ?name ?value
WHERE {
  ?airport wdt:P31/wdt:P279* wd:Q1248784 ;
           rdfs:label ?name .
  FILTER(LANG(?name) = "en")
  OPTIONAL {
    ?airport wdt:P131 ?loc .
    ?loc wdt:P421 ?tz .
    OPTIONAL { ?tz rdfs:label ?label . FILTER(LANG(?label) = "en") }
    BIND(COALESCE(?label, STR(?tz)) AS ?value)
  }
}
"""

# Elevation is a special case: wdt:P2044 (the "truthy" shorthand) returns only the bare
# number, stripping the unit. In practice this data mixes feet and metres (verified: 11,825
# statements in feet vs. 6,942 in metres across the dataset), which silently produces
# nonsense values if treated as all-metres (e.g. a 4,300 m airport misread as 14,108).
# So we query the full statement node (p:P2044/psv:P2044) to get the unit alongside the
# amount, and convert to metres ourselves.
ELEVATION_QUERY = """
PREFIX wd:       <http://www.wikidata.org/entity/>
PREFIX wdt:      <http://www.wikidata.org/prop/direct/>
PREFIX p:        <http://www.wikidata.org/prop/>
PREFIX psv:      <http://www.wikidata.org/prop/statement/value/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?airport ?name ?amount ?unit
WHERE {
  ?airport wdt:P31/wdt:P279* wd:Q1248784 ;
           rdfs:label ?name .
  FILTER(LANG(?name) = "en")
  OPTIONAL {
    ?airport p:P2044 ?stmt .
    ?stmt psv:P2044 ?val .
    ?val wikibase:quantityAmount ?amount ;
         wikibase:quantityUnit ?unit .
  }
}
"""

METRE_UNIT = "http://www.wikidata.org/entity/Q11573"
FOOT_UNIT = "http://www.wikidata.org/entity/Q3710"
FOOT_TO_METRE = 0.3048


def elevation_to_metres(amount, unit):
    if pd.isna(amount) or pd.isna(unit):
        return None
    if unit == METRE_UNIT:
        return float(amount)
    if unit == FOOT_UNIT:
        return float(amount) * FOOT_TO_METRE
    return None  # unrecognised unit (e.g. a bare/dimensionless statement) -- don't guess

In [3]:
frames = []
for attr_name, prop_name in LITERAL_ATTRIBUTES.items():
    print(f"Fetching {attr_name} ({prop_name})...")
    df = run_sparql_query(build_literal_query(prop_name), endpoint=WIKIDATA_ENDPOINT)
    df["attribute"] = attr_name
    frames.append(df)

for attr_name, prop_name in ENTITY_ATTRIBUTES.items():
    print(f"Fetching {attr_name} ({prop_name}, resolved to label)...")
    df = run_sparql_query(build_entity_query(prop_name), endpoint=WIKIDATA_ENDPOINT)
    df["attribute"] = attr_name
    frames.append(df)

print("Fetching timezone (wdt:P131 -> wdt:P421, resolved to label)...")
timezone_df = run_sparql_query(TIMEZONE_QUERY, endpoint=WIKIDATA_ENDPOINT)
timezone_df["attribute"] = "timezone"
frames.append(timezone_df)

print("Fetching elevation (p:P2044 with unit, converted to metres)...")
elevation_raw_df = run_sparql_query(ELEVATION_QUERY, endpoint=WIKIDATA_ENDPOINT)
elevation_df = elevation_raw_df[["airport", "name"]].copy()
elevation_df["value"] = [
    elevation_to_metres(amount, unit)
    for amount, unit in zip(elevation_raw_df["amount"], elevation_raw_df["unit"])
]
elevation_df["attribute"] = "elevation"
frames.append(elevation_df)

raw_long_df = pd.concat(frames, ignore_index=True)

print("Fetching coordinates (wdt:P625 via geof:latitude/longitude)...")
raw_coord_df = run_sparql_query(COORD_QUERY, endpoint=WIKIDATA_ENDPOINT)

print("raw_long_df:", raw_long_df.shape)
print("raw_coord_df:", raw_coord_df.shape)

Fetching iata (wdt:P238)...
Fetching icao (wdt:P239)...
Fetching faa (wdt:P240)...
Fetching located_in (wdt:P131, resolved to label)...
Fetching country (wdt:P17, resolved to label)...
Fetching operator (wdt:P137, resolved to label)...
Fetching owner (wdt:P127, resolved to label)...
Fetching timezone (wdt:P131 -> wdt:P421, resolved to label)...
Fetching elevation (p:P2044 with unit, converted to metres)...
Fetching coordinates (wdt:P625 via geof:latitude/longitude)...
raw_long_df: (247303, 4)
raw_coord_df: (31404, 4)


In [4]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
raw_long_df.to_csv(RAW_DIR / "wikidata_airports_raw_long.csv", index=False)
raw_coord_df.to_csv(RAW_DIR / "wikidata_airports_raw_coordinates.csv", index=False)
print(f"Saved raw data to {RAW_DIR}")

Saved raw data to /Users/huanglinchun/Desktop/Master/Studies/Web-Data-Integration/Web-Data-Integration-Project/data/raw/wikidata


## 2. Conflict resolution

**Strategy**: per airport, prefer the *mode* (most common value); if every value is unique (no repeats — typical for numeric attributes), fall back to the first value encountered.

This is a simple, generic tie-breaker applied uniformly. It's a reasonable default for the low-conflict attributes (`country`, `located_in`, `iata`, `icao`, `faa`, `elevation`, `operator`, `owner` — all under 4% conflict per the profiling notebook).

**Coordinates are resolved as a `(lat, lon)` *pair*, not per-axis.** Resolving latitude and longitude independently would let the chosen latitude come from one source statement and the chosen longitude from another, producing a `(lat, lon)` combination that no source ever asserted (verified: ~2,500 airports would get such a fabricated point). `resolve_coord_pair` picks the most common exact pair instead (or the first pair seen), so the result is always a real, source-attested location. For the ~21.5% of airports with conflicting coordinates this is still a **provisional** tie-break — a handful have genuinely far-apart pairs (a real data conflict, not precision noise) that deserve a closer look before fusion — but it can no longer invent a coordinate.

`timezone` (~28.3% conflict) is likewise resolved provisionally with the generic mode/first rule; see the "Open item" in `wikidata_profiling.ipynb`.

In [5]:
from collections import Counter


def resolve_single_value(series: pd.Series):
    """Pick one representative value per airport: mode if there's a repeated value,
    otherwise the first value seen."""
    values = series.dropna()
    if values.empty:
        return None
    modes = values.mode()
    return modes.iloc[0] if len(modes) else values.iloc[0]


def resolve_coord_pair(group: pd.DataFrame) -> pd.Series:
    """Resolve one (lat, lon) pair per airport as a unit, never per-axis.

    Picks the most common exact (lat, lon) pair; if every pair is distinct
    (typical precision noise), falls back to the first pair seen. Guarantees
    the result is a coordinate that actually appears in the source data,
    instead of mixing a latitude from one statement with a longitude from
    another."""
    pairs = [
        (lat, lon)
        for lat, lon in zip(group["lat"], group["lon"])
        if pd.notna(lat) and pd.notna(lon)
    ]
    if not pairs:
        return pd.Series({"lat": None, "lon": None})
    lat, lon = Counter(pairs).most_common(1)[0][0]
    return pd.Series({"lat": lat, "lon": lon})

In [6]:
final_df = raw_long_df.groupby("airport")["name"].first().to_frame()

for attr_name, group in raw_long_df.groupby("attribute"):
    resolved = group.groupby("airport")["value"].apply(resolve_single_value)
    final_df = final_df.join(resolved.rename(attr_name), how="left")

coord_resolved = raw_coord_df.groupby("airport")[["lat", "lon"]].apply(resolve_coord_pair)
final_df = final_df.join(coord_resolved, how="left")

final_df = final_df.reset_index()
print(final_df.shape)
final_df.head()

(24871, 13)


,airport,name,country,elevation,faa,iata,icao,located_in,operator,owner,timezone,lat,lon
0,http://www.wikidata.org/entity/Q1000018,Ji'an Jinggangshan Airport,People's Republic of China,86.0000,None,JGS,ZSGS,Taihe County,None,None,None,26.85749994298,114.7372220966
1,http://www.wikidata.org/entity/Q1000025,Changde Taohuayuan Airport,People's Republic of China,NaN,None,CGD,ZGCD,Hunan,None,None,None,28.91889994863,111.6399999071
2,http://www.wikidata.org/entity/Q1002604,Jiujiang Lushan Airport,People's Republic of China,49.9872,None,JIU,ZSJJ,Jiujiang,None,None,None,29.47694443118,115.8011112509
3,http://www.wikidata.org/entity/Q1006478,Aksu Hongqipo Airport,People's Republic of China,1163.0000,None,AKU,ZWAK,Hongqipo,None,None,None,41.26249994269,80.29166659367
4,http://www.wikidata.org/entity/Q1006900,Mohe Gulian Airport,People's Republic of China,579.0000,None,OHE,ZYMH,Heilongjiang,None,None,None,52.92111102764,122.4205556721


In [7]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(INTERIM_DIR / "wikidata_airports.csv", index=False)
print(f"Saved final dataset to {INTERIM_DIR / 'wikidata_airports.csv'} ({len(final_df)} airports)")

Saved final dataset to /Users/huanglinchun/Desktop/Master/Studies/Web-Data-Integration/Web-Data-Integration-Project/data/interim/wikidata_airports.csv (24871 airports)


## Notes

- Final row count (**24,237**) is slightly below the 24,434 airports profiled in `wikidata_profiling.ipynb` — the profiling queries didn't require a name, but this notebook needs one (for a usable dataset), so the ~197 airports (0.8%) without an English `rdfs:label` get dropped here. Worth deciding later whether to fall back to another language's label for those instead of dropping them.
- `coordinates` and `timezone` conflict resolution is provisional (see above) — revisit before treating this file as final.
- `elevation` is fetched via the full statement node (not `wdt:P2044` directly) and converted to metres, because the raw data mixes feet and metres with no way to tell them apart from the bare number alone — this was caught by `wikidata_eda.ipynb` surfacing ~306 airports with physically impossible elevations (e.g. 14,108, which is a ~4,300 m airport misread as feet).
- `located_in`, `country`, `operator`, `owner`, and `timezone` are entity-valued properties (they point to another Wikidata item, not a literal) — resolved to the entity's English label (with a raw-URI fallback via `COALESCE` if no English label exists) so the dataset holds readable text instead of Q-ID URIs. `airport` itself stays as a URI on purpose: it's the stable join key, and `name` already carries the readable form.